# Deep Learning Experiment Notebook (BiLSTM)

**Project:** `pba2026-Nexus`  
**Task:** Advanced Sentiment Analysis with Deep Learning (PyTorch)


## 1) Objective

Notebook ini mendokumentasikan eksperimen model **BiLSTM** untuk klasifikasi sentimen tweet.

Checklist ketentuan tugas:

- Menentukan arsitektur DL ✅
- Verifikasi parameter model ≤ 10 juta ✅
- Implementasi model dengan PyTorch ✅
- Latih model + catat train/validation loss ✅
- Evaluasi model pada data uji ✅
- Siapkan demo Streamlit model DL ✅
- Siapkan deploy ke Hugging Face Spaces ✅


In [ ]:
# (Opsional) Install dependency jika belum ada
# !pip install torch pandas numpy matplotlib scikit-learn tqdm

import json
import random
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 2) Konfigurasi Eksperimen

In [ ]:
SEED = 42
DATA_PATH = Path('../data/cleaned_sample.csv')

# Split
TEST_SIZE = 0.1
VAL_SIZE = 0.1

# Vocabulary / sequence
MAX_VOCAB_SIZE = 20_000
MIN_FREQ = 2
MAX_LEN = 50

# Model
EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.3
NUM_CLASSES = 2
BIDIRECTIONAL = True
MAX_ALLOWED_PARAMS = 10_000_000

# Train
BATCH_SIZE = 64
EPOCHS = 6
LR = 1e-3
WEIGHT_DECAY = 1e-4

# Output
RUN_NAME = 'bilstm_sentiment_v1'
MODEL_DIR = Path('../models/dl')
ASSETS_DIR = Path('../assets/dl')

MODEL_DIR.mkdir(parents=True, exist_ok=True)
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 3) Load Data

In [ ]:
assert DATA_PATH.exists(), f'Dataset not found: {DATA_PATH}'
df = pd.read_csv(DATA_PATH)
df = df[['clean_text', 'sentiment']].dropna().copy()
df['clean_text'] = df['clean_text'].astype(str)
df['sentiment'] = df['sentiment'].astype(int)

print('Total data:', len(df))
print(df.head())
print('Label distribution:', Counter(df['sentiment'].tolist()))

## 4) Tokenizer dan Vocabulary

In [ ]:
def basic_tokenize(text: str) -> List[str]:
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    if not text:
        return []
    return text.split(' ')

def build_vocab(texts: List[str], min_freq=2, max_vocab_size=20_000):
    counter = Counter()
    for t in texts:
        counter.update(basic_tokenize(t))

    stoi = {'<PAD>': 0, '<UNK>': 1}
    for token, freq in counter.most_common():
        if freq < min_freq:
            continue
        if token in stoi:
            continue
        if len(stoi) >= max_vocab_size:
            break
        stoi[token] = len(stoi)

    return stoi

def encode_text(text: str, stoi: Dict[str, int], max_len: int) -> Tuple[List[int], int]:
    tokens = basic_tokenize(text)
    ids = [stoi.get(tok, stoi['<UNK>']) for tok in tokens][:max_len]
    length = len(ids)
    if length < max_len:
        ids += [stoi['<PAD>']] * (max_len - length)
    return ids, length

## 5) Split Data (Train/Val/Test)

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df['sentiment']
)

val_rel_size = VAL_SIZE / (1.0 - TEST_SIZE)
train_df, val_df = train_test_split(
    train_df,
    test_size=val_rel_size,
    random_state=SEED,
    stratify=train_df['sentiment']
)

print('Train:', len(train_df))
print('Val  :', len(val_df))
print('Test :', len(test_df))

In [ ]:
stoi = build_vocab(
    texts=train_df['clean_text'].tolist(),
    min_freq=MIN_FREQ,
    max_vocab_size=MAX_VOCAB_SIZE
)
print('Vocab size:', len(stoi))

## 6) Dataset dan DataLoader

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, stoi, max_len):
        self.texts = texts
        self.labels = labels
        self.stoi = stoi
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = int(self.labels[idx])
        input_ids, length = encode_text(text, self.stoi, self.max_len)
        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(length, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

train_ds = SentimentDataset(train_df['clean_text'].tolist(), train_df['sentiment'].tolist(), stoi, MAX_LEN)
val_ds   = SentimentDataset(val_df['clean_text'].tolist(), val_df['sentiment'].tolist(), stoi, MAX_LEN)
test_ds  = SentimentDataset(test_df['clean_text'].tolist(), test_df['sentiment'].tolist(), stoi, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

len(train_loader), len(val_loader), len(test_loader)

## 7) Arsitektur BiLSTM (PyTorch)

In [ ]:
@dataclass
class BiLSTMConfig:
    vocab_size: int
    embedding_dim: int = 128
    hidden_dim: int = 128
    num_layers: int = 2
    dropout: float = 0.3
    num_classes: int = 2
    pad_idx: int = 0
    bidirectional: bool = True

class BiLSTMClassifier(nn.Module):
    def __init__(self, config: BiLSTMConfig):
        super().__init__()
        self.config = config
        self.embedding = nn.Embedding(
            num_embeddings=config.vocab_size,
            embedding_dim=config.embedding_dim,
            padding_idx=config.pad_idx,
        )

        lstm_dropout = config.dropout if config.num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=config.embedding_dim,
            hidden_size=config.hidden_dim,
            num_layers=config.num_layers,
            batch_first=True,
            dropout=lstm_dropout,
            bidirectional=config.bidirectional,
        )

        out_dim = config.hidden_dim * (2 if config.bidirectional else 1)
        self.dropout = nn.Dropout(config.dropout)
        self.classifier = nn.Linear(out_dim, config.num_classes)

    def forward(self, input_ids, lengths=None):
        emb = self.embedding(input_ids)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                emb, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            _, (h_n, _) = self.lstm(packed)
        else:
            _, (h_n, _) = self.lstm(emb)

        if self.config.bidirectional:
            forward_last = h_n[-2]
            backward_last = h_n[-1]
            feat = torch.cat([forward_last, backward_last], dim=1)
        else:
            feat = h_n[-1]

        feat = self.dropout(feat)
        logits = self.classifier(feat)
        return logits

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

cfg = BiLSTMConfig(
    vocab_size=len(stoi),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    num_classes=NUM_CLASSES,
    pad_idx=stoi['<PAD>'],
    bidirectional=BIDIRECTIONAL,
)

model = BiLSTMClassifier(cfg).to(device)
n_params = count_trainable_params(model)
print('Trainable params:', f'{n_params:,}')
print('Max allowed     :', f'{MAX_ALLOWED_PARAMS:,}')
print('Status          :', 'PASS ✅' if n_params <= MAX_ALLOWED_PARAMS else 'FAIL ❌')

## 8) Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def run_epoch_train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0

    for input_ids, lengths, labels in tqdm(loader, desc='Train', leave=False):
        input_ids = input_ids.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(input_ids, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / max(total, 1), total_correct / max(total, 1)

@torch.no_grad()
def run_epoch_eval(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0

    for input_ids, lengths, labels in tqdm(loader, desc='Eval', leave=False):
        input_ids = input_ids.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)

        logits = model(input_ids, lengths)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / max(total, 1), total_correct / max(total, 1)

history = []
best_val_loss = float('inf')
best_state_dict = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch_train(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = run_epoch_eval(model, val_loader, criterion, device)

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc,
    })

    print(f"[Epoch {epoch:02d}/{EPOCHS}] train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}

if best_state_dict is not None:
    model.load_state_dict(best_state_dict)

history_df = pd.DataFrame(history)
history_df.head()

## 9) Plot Kurva Training/Validation Loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_df['epoch'], history_df['train_loss'], label='train_loss')
plt.plot(history_df['epoch'], history_df['val_loss'], label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('BiLSTM Training / Validation Loss')
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

history_csv_path = ASSETS_DIR / f'{RUN_NAME}_history.csv'
loss_plot_path = ASSETS_DIR / f'{RUN_NAME}_loss_curve.png'

history_df.to_csv(history_csv_path, index=False)
plt.figure(figsize=(8, 5))
plt.plot(history_df['epoch'], history_df['train_loss'], label='train_loss')
plt.plot(history_df['epoch'], history_df['val_loss'], label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('BiLSTM Training / Validation Loss')
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(loss_plot_path, dpi=150)
plt.close()

print('Saved history:', history_csv_path)
print('Saved loss plot:', loss_plot_path)

## 10) Evaluasi Model pada Test Set

In [ ]:
@torch.no_grad()
def predict_loader(model, loader, device):
    model.eval()
    y_true, y_pred = [], []

    for input_ids, lengths, labels in tqdm(loader, desc='Predict', leave=False):
        input_ids = input_ids.to(device)
        lengths = lengths.to(device)

        logits = model(input_ids, lengths)
        preds = torch.argmax(logits, dim=1).cpu().numpy().tolist()

        y_pred.extend(preds)
        y_true.extend(labels.numpy().tolist())

    return y_true, y_pred

test_loss, test_acc = run_epoch_eval(model, test_loader, criterion, device)
y_true, y_pred = predict_loader(model, test_loader, device)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='binary', zero_division=0)
rec = recall_score(y_true, y_pred, average='binary', zero_division=0)
f1 = f1_score(y_true, y_pred, average='binary', zero_division=0)
cm = confusion_matrix(y_true, y_pred)

print('=== Test Metrics ===')
print('Loss      :', round(test_loss, 4))
print('Accuracy  :', round(acc, 4))
print('Precision :', round(prec, 4))
print('Recall    :', round(rec, 4))
print('F1-score  :', round(f1, 4))
print('Confusion Matrix:\n', cm)

## 11) Simpan Artifact Model

In [ ]:
model_dir = MODEL_DIR
model_dir.mkdir(parents=True, exist_ok=True)

state_dict_path = model_dir / 'bilstm_state_dict.pt'
checkpoint_path = model_dir / 'bilstm_sentiment.pt'
vocab_path = model_dir / 'vocab.json'
train_config_path = model_dir / 'train_config.json'
deploy_config_path = model_dir / 'config.json'
train_metrics_path = model_dir / 'train_metrics.json'
eval_metrics_path = model_dir / 'eval_metrics.json'

torch.save(model.state_dict(), state_dict_path)

checkpoint_obj = {
    'state_dict': model.state_dict(),
    'model_config': {
        'vocab_size': len(stoi),
        'embedding_dim': EMBEDDING_DIM,
        'hidden_dim': HIDDEN_DIM,
        'num_layers': NUM_LAYERS,
        'dropout': DROPOUT,
        'num_classes': NUM_CLASSES,
        'pad_idx': stoi['<PAD>'],
        'bidirectional': BIDIRECTIONAL,
    },
    'stoi': stoi,
    'max_len': MAX_LEN,
    'label_map': {'0': 'negative', '1': 'positive'},
    'run_name': RUN_NAME,
}
torch.save(checkpoint_obj, checkpoint_path)

with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump(stoi, f, ensure_ascii=False, indent=2)

config_obj = {
    'vocab_size': len(stoi),
    'embedding_dim': EMBEDDING_DIM,
    'hidden_dim': HIDDEN_DIM,
    'num_layers': NUM_LAYERS,
    'dropout': DROPOUT,
    'num_classes': NUM_CLASSES,
    'pad_token': '<PAD>',
    'unk_token': '<UNK>',
    'pad_idx': stoi['<PAD>'],
    'unk_idx': stoi['<UNK>'],
    'max_len': MAX_LEN,
    'bidirectional': BIDIRECTIONAL,
    'label_map': {'0': 'negative', '1': 'positive'},
    'run_name': RUN_NAME,
}

with open(train_config_path, 'w', encoding='utf-8') as f:
    json.dump(config_obj, f, ensure_ascii=False, indent=2)
with open(deploy_config_path, 'w', encoding='utf-8') as f:
    json.dump(config_obj, f, ensure_ascii=False, indent=2)

train_metrics = {
    'run_name': RUN_NAME,
    'best_val_loss': float(best_val_loss),
    'test_loss': float(test_loss),
    'test_acc': float(test_acc),
    'trainable_params': int(n_params),
    'max_allowed_params': int(MAX_ALLOWED_PARAMS),
    'parameter_limit_ok': bool(n_params <= MAX_ALLOWED_PARAMS),
}

eval_metrics = {
    'num_test_samples': len(y_true),
    'accuracy': float(acc),
    'precision': float(prec),
    'recall': float(rec),
    'f1_score': float(f1),
    'confusion_matrix': cm.tolist(),
}

with open(train_metrics_path, 'w', encoding='utf-8') as f:
    json.dump(train_metrics, f, ensure_ascii=False, indent=2)
with open(eval_metrics_path, 'w', encoding='utf-8') as f:
    json.dump(eval_metrics, f, ensure_ascii=False, indent=2)

print('Saved:', state_dict_path)
print('Saved:', checkpoint_path)
print('Saved:', vocab_path)
print('Saved:', train_config_path)
print('Saved:', deploy_config_path)
print('Saved:', train_metrics_path)
print('Saved:', eval_metrics_path)

## 12) Ringkasan Hasil

- Arsitektur: **BiLSTM (PyTorch)**
- Batas parameter: **PASS** (≤ 10 juta)
- Artifact training dan evaluasi sudah tersimpan di `models/dl/`
- Kurva loss tersimpan di `assets/dl/`
- Demo lokal tersedia di `app/app_dl.py`
- Paket deploy tersedia di `huggingface_app_dl/`
